# Teoria LLM gateway — OpenAI-compatible API

Este notebook cobre todos os recursos da interface OpenAI-compatible do gateway:

- **`GET /v1/models`** — listar modelos disponíveis via OpenAI SDK e httpx.
- **`POST /v1/chat/completions`** (não-stream e stream) via `langchain_openai.ChatOpenAI` e `openai.OpenAI`.
- **`POST /api/v1/chat`** (API simplificada) via `httpx`.

O gateway é compatível com qualquer cliente que fale OpenAI: SDK oficial, LangChain, LlamaIndex, etc.

## Variáveis de ambiente

| Variável | Significado | Padrão |
|----------|-------------|--------|
| `TEORIA_BASE_URL` | URL base do gateway (sem barra final). GPU via túnel SSH: `http://localhost:8080`. Stack local mock: `http://localhost:8081`. Produção: `https://llm.jambu.ai`. | `http://localhost:8080` |
| `TEORIA_API_KEY` | Chave (`Authorization: Bearer …` ou `x-api-key`). | `default` |
| `TEORIA_MODEL` | Nome do modelo enviado pelo cliente; o gateway pode sobrescrever com `VLLM_MODEL`. | `gpt-4o-mini` |

Instalação: `pip install -r notebooks/requirements.txt` (ou `uv pip install -r notebooks/requirements.txt`).


In [26]:
!uv pip install -r requirements.txt

Resolved 138 packages in 1.13s                                       
Prepared 20 packages in 2.45s                                            
Installed 138 packages in 840ms                             
 + annotated-types==0.7.0
 + anthropic==0.86.0
 + anyio==4.13.0
 + argon2-cffi==25.1.0
 + argon2-cffi-bindings==25.1.0
 + arrow==1.4.0
 + asttokens==3.0.1
 + async-lru==2.3.0
 + attrs==26.1.0
 + babel==2.18.0
 + beautifulsoup4==4.14.3
 + bleach==6.3.0
 + bracex==2.6
 + certifi==2026.2.25
 + cffi==2.0.0
 + charset-normalizer==3.4.6
 + comm==0.2.3
 + cryptography==46.0.5
 + debugpy==1.8.20
 + decorator==5.2.1
 + deepagents==0.4.12
 + defusedxml==0.7.1
 + distro==1.9.0
 + docstring-parser==0.17.0
 + executing==2.2.1
 + fastjsonschema==2.21.2
 + filetype==1.2.0
 + fqdn==1.5.1
 + google-auth==2.49.1
 + google-genai==1.68.0
 + h11==0.16.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + idna==3.11
 + ipykernel==7.2.0
 + ipython==9.11.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + isodura

In [19]:
import json
import os

import httpx
import openai
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

# GPU via túnel SSH: http://localhost:8080
# Stack mock local (make test-up): http://localhost:8081
# Produção: https://llm.jambu.ai
TEORIA_BASE = os.environ.get("TEORIA_BASE_URL", "http://localhost:8080").rstrip("/")
TEORIA_API_KEY = os.environ.get(
    "TEORIA_API_KEY", os.environ.get("GATEWAY_API_KEY", "test-key-gpu-validation")
)
# LangChain sempre envia um campo model; o gateway usa VLLM_MODEL para substituir.
TEORIA_MODEL = os.environ.get("TEORIA_MODEL", "gpt-4o-mini")

OPENAI_COMPAT_BASE = f"{TEORIA_BASE}/v1"
SIMPLIFIED_CHAT_URL = f"{TEORIA_BASE}/api/v1/chat"

print("TEORIA_BASE         :", TEORIA_BASE)
print("OpenAI-compat base  :", OPENAI_COMPAT_BASE)
print("chat/completions    :", f"{OPENAI_COMPAT_BASE}/chat/completions")
print("models              :", f"{OPENAI_COMPAT_BASE}/models")
print("simplified chat     :", SIMPLIFIED_CHAT_URL)


TEORIA_BASE         : http://localhost:18080
OpenAI-compat base  : http://localhost:18080/v1
chat/completions    : http://localhost:18080/v1/chat/completions
models              : http://localhost:18080/v1/models
simplified chat     : http://localhost:18080/api/v1/chat


## 0) `GET /v1/models` — listar modelos disponíveis

O gateway expõe `GET /v1/models` compatível com a OpenAI API. Qualquer cliente que precise descobrir o modelo antes de inferir pode usar esse endpoint — inclusive o SDK oficial e o LangChain.

In [20]:
# --- via OpenAI SDK (client.models.list) ------------------------------------
client = openai.OpenAI(base_url=OPENAI_COMPAT_BASE, api_key=TEORIA_API_KEY)

models = client.models.list()
print("OpenAI SDK — modelos disponíveis:")
for m in models.data:
    print(f"  id={m.id}  owned_by={m.owned_by}")

# Atualiza TEORIA_MODEL com o primeiro modelo real retornado pelo servidor
if models.data:
    TEORIA_MODEL = models.data[0].id
    print(f"\nTEORIA_MODEL atualizado para: {TEORIA_MODEL}")

# --- via httpx (equivalente curl GET /v1/models) ----------------------------
with httpx.Client(timeout=30.0) as http:
    r = http.get(
        f"{OPENAI_COMPAT_BASE}/models",
        headers={"Authorization": f"Bearer {TEORIA_API_KEY}"},
    )
    r.raise_for_status()
    data = r.json()
    print(f"\nhttpx — {len(data['data'])} modelo(s) na lista (object={data['object']})")

OpenAI SDK — modelos disponíveis:
  id=nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16  owned_by=vllm

TEORIA_MODEL atualizado para: nvidia/NVIDIA-Nemotron-3-Nano-4B-BF16

httpx — 1 modelo(s) na lista (object=list)


## 1) OpenAI-compatible — conclusão única (README: `curl` sem `stream`)

Equivalente a `messages` + `max_tokens` em `/v1/chat/completions`.


In [21]:
llm = ChatOpenAI(
    base_url=OPENAI_COMPAT_BASE,
    api_key=TEORIA_API_KEY,
    model=TEORIA_MODEL,
    max_tokens=512,
    temperature=0
    
)

response = llm.invoke([HumanMessage(content="Hello there")])
print(response.content)


We need to respond. The user says "Hello there". We should respond politely. No special instructions. Just a friendly greeting.
</think>
Hello! How can I assist you today?


## 2) OpenAI-compatible — streaming (`stream: true`)

LangChain usa o mesmo endpoint; o cliente HTTP negocia SSE como no `curl -N` do README.


In [22]:
llm = ChatOpenAI(
    base_url=OPENAI_COMPAT_BASE,
    api_key=TEORIA_API_KEY,
    model=TEORIA_MODEL,
    max_tokens=512,
    temperature=0
)

for chunk in llm.stream([HumanMessage(content="Hello")]):
    if chunk.content:
        print(chunk.content, end="", flush=True)
print()


We need to respond. The user says "Hello". We should greet back. Probably a simple response.
</think>
Hello! How can I assist you today?


## 3) API simplificada `/api/v1/chat` (README: `input` + `stream`)

**Compatibilidade LangChain:** `ChatOpenAI` fala apenas com rotas estilo OpenAI (`/v1/chat/completions`). O contrato `input` / `system_prompt` é outro path; aqui usamos `httpx`. A resposta **não-stream** do gateway ainda é JSON no formato OpenAI (`choices[0].message.content`), o que facilita o parse manual.


In [23]:
headers = {
    "Authorization": f"Bearer {TEORIA_API_KEY}",
    "Content-Type": "application/json",
}

with httpx.Client(timeout=120.0) as http:
    r = http.post(
        SIMPLIFIED_CHAT_URL,
        json={
            "input": "What is 2+2? Reply with one word.",
            "max_tokens": 32,
            "stream": False,
        },
        headers=headers,
    )
    r.raise_for_status()
    body = r.json()
    print("non-stream:", body["choices"][0]["message"]["content"])

    with http.stream(
        "POST",
        SIMPLIFIED_CHAT_URL,
        json={
            "input": "Count from 1 to 10, one per line",
            "max_tokens": 100,
            "stream": True,
        },
        headers=headers,
    ) as resp:
        resp.raise_for_status()
        pieces: list[str] = []
        for line in resp.iter_lines():
            if not line or not line.startswith("data: "):
                continue
            payload = line[6:]
            if payload == "[DONE]":
                break
            chunk = json.loads(payload)
            delta = chunk["choices"][0].get("delta") or {}
            piece = delta.get("content") or ""
            if piece:
                pieces.append(piece)
                print(piece, end="", flush=True)
        print("\nstream chars:", sum(len(p) for p in pieces))


non-stream: The user asks: "What is 2+2? Reply with one word." The answer is "four". Must reply with one word only. So output
User asks: "Count from 1 to 10, one per line". So output 1 to 10 each on its own line. Simple.
</think>
1  
2  
3  
4  
5  
6  
7  
8  
9  
10
stream chars: 142


## Notas de compatibilidade

| Recurso | Status | Detalhe |
|---------|--------|---------|
| `GET /v1/models` | ✅ | Proxied ao backend; retorna lista OpenAI-padrão. Auth obrigatório. |
| `POST /v1/chat/completions` | ✅ | Non-stream e stream SSE. Aceita `max_tokens` e `max_completion_tokens`. |
| `POST /api/v1/chat` | ✅ | API simplificada (`input` + `system_prompt`). Fora do path OpenAI. |
| Erros | ✅ | Envelope `{"error": {"message", "type", "code"}}` — compatível com `openai.BadRequestError`. |
| Auth | ✅ | `Authorization: Bearer` e `x-api-key` ambos aceitos. |
| Streaming | ✅ | SSE: `data: {json}` por chunk, `data: [DONE]` ao final. |

**Dicas práticas:**

1. **`model` obrigatório no cliente** — o construtor `ChatOpenAI` e `openai.OpenAI` exigem um nome. Use `GET /v1/models` para descobrir o ID real e evitar divergências.
2. **`/api/v1/chat` fora do ecossistema OpenAI** — use `httpx` ou um `RunnableLambda` se precisar de `input` + `system_prompt`; o `ChatOpenAI` fala apenas com `/v1/chat/completions`.
3. **Túnel SSH** — com `ssh -L 8080:localhost:8080 ...` o gateway fica em `http://localhost:8080`; defina `TEORIA_BASE_URL=http://localhost:8080` antes de executar o notebook.


## 4) Deep Agent com `deepagents`

`deepagents` é a framework de agentes da LangChain — construída sobre LangGraph — com planejamento de tarefas, sistema de arquivos virtual, subagentes e memória de longo prazo integrados. Qualquer `BaseChatModel` do LangChain pode ser usado como motor, inclusive o `ChatOpenAI` apontado para o nosso gateway.

**Requisito:** o modelo precisa suportar **tool calling** (function calling). O vLLM expõe isso via a OpenAI API. A qualidade das chamadas depende do treinamento do modelo; modelos menores (como o Nemotron 4B) podem ter comportamento menos consistente do que modelos maiores treinados especificamente para agentes.

Instale se necessário: `pip install deepagents>=0.4.12`

In [27]:
import datetime
import math

from deepagents import create_deep_agent
from langchain_openai import ChatOpenAI

# Modelo apontado para o nosso gateway OpenAI-compatible
llm_agent = ChatOpenAI(
    base_url=OPENAI_COMPAT_BASE,
    api_key=TEORIA_API_KEY,
    model=TEORIA_MODEL,
    max_tokens=1024,
    temperature=0,
)

# ------------------------------------------------------------------
# Ferramentas simples que o agente pode usar
# ------------------------------------------------------------------

def get_current_datetime() -> str:
    """Return the current UTC date and time as an ISO-8601 string."""
    return datetime.datetime.utcnow().isoformat() + "Z"


def calculate(expression: str) -> str:
    """Safely evaluate a basic mathematical expression and return the result.

    Supports: +, -, *, /, **, sqrt, abs, round, int, float, math.*
    Examples: '2 ** 10', 'math.sqrt(144)', 'round(3.14159, 2)'
    """
    allowed = {k: getattr(math, k) for k in dir(math) if not k.startswith("_")}
    allowed.update({"abs": abs, "round": round, "int": int, "float": float})
    try:
        result = eval(expression, {"__builtins__": {}}, allowed)  # noqa: S307
        return str(result)
    except Exception as exc:
        return f"Error: {exc}"


def list_gateway_models() -> str:
    """List models available on the Teoria LLM gateway."""
    import httpx
    with httpx.Client(timeout=10.0) as http:
        r = http.get(
            f"{OPENAI_COMPAT_BASE}/models",
            headers={"Authorization": f"Bearer {TEORIA_API_KEY}"},
        )
        r.raise_for_status()
        data = r.json()
    models = [m["id"] for m in data.get("data", [])]
    return f"Available models: {', '.join(models)}"


# ------------------------------------------------------------------
# Criação do agente
# ------------------------------------------------------------------

agent = create_deep_agent(
    name="teoria-agent",
    model=llm_agent,
    tools=[get_current_datetime, calculate, list_gateway_models],
    system_prompt=(
        "You are a helpful assistant running on the Teoria LLM gateway. "
        "Use the available tools when needed. Be concise and accurate."
    ),
)

print("Agent created:", type(agent).__name__)

Agent created: CompiledStateGraph


### 4a) `invoke` — resposta única com uso de ferramenta

In [28]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "What is 2 ** 32? Also tell me the current UTC time."}]
})

# A última mensagem no histórico é a resposta final do agente
final_message = result["messages"][-1]
print(final_message.content)

BadRequestError: Error code: 400 - {'error': {'message': '"auto" tool choice requires --enable-auto-tool-choice and --tool-call-parser to be set', 'type': 'BadRequestError', 'param': None, 'code': 400}}

### 4b) `stream` — atualizações em tempo real por nó do grafo LangGraph

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

# stream_mode="updates" entrega um dict por nó executado no grafo LangGraph.
# Cada update contém as novas mensagens geradas por aquele nó.
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "List the models available on the gateway, then calculate sqrt(1764)."}]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        messages = update.get("messages", [])
        for msg in messages:
            if isinstance(msg, AIMessage):
                # Tool calls planejados pelo agente
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"[{node_name}] → tool call: {tc['name']}({tc['args']})")
                # Resposta textual final
                elif msg.content:
                    print(f"[{node_name}] → {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"[tool:{msg.name}] ← {str(msg.content)[:120]}")